# Telco Churn - Modeling and Evaluation

In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from imblearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    PrecisionRecallDisplay,
    RocCurveDisplay,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, cross_val_predict

from src.models import (
    clone_for_resampling,
    cross_validate_model,
    evaluate_on_test,
    find_optimal_threshold_from_proba,
    get_hyperparam_grid,
    get_model_configs,
    get_resampling_strategies,
    load_processed_data,
)

## Load processed data

In [ ]:
x_train, x_test, y_train, y_test = load_processed_data("../data/processed")

print("x_train:", x_train.shape)
print("x_test:", x_test.shape)
print("y_train balance:")
print(y_train.value_counts(normalize=True))
print("y_test balance:")
print(y_test.value_counts(normalize=True))

## Imbalance strategy comparison (CV)

Quick pass comparing baseline, class-weight balancing, SMOTE, and SMOTE+Tomek.

In [ ]:
all_configs = get_model_configs()
base_models = {k: all_configs[k] for k in ["logistic_regression", "random_forest", "lightgbm"]}
balanced_models = {
    k: all_configs[k]
    for k in ["logistic_regression_balanced", "random_forest_balanced", "lightgbm_balanced"]
}
resampling = get_resampling_strategies()

results = []

# Baseline + SMOTE strategies for base models.
for model_name, model in base_models.items():
    for strategy_name, sampler in resampling.items():
        model_for_run = clone_for_resampling(model, strategy_name)
        estimator = (
            Pipeline([("sampler", sampler), ("model", model_for_run)])
            if sampler is not None
            else model_for_run
        )
        summary = cross_validate_model(estimator, x_train, y_train, cv=5)
        summary.update({"model": model_name, "strategy": strategy_name})
        results.append(summary)

# Class-weight strategy from balanced variants.
for model_name, model in balanced_models.items():
    summary = cross_validate_model(model, x_train, y_train, cv=5)
    summary.update({"model": model_name.replace("_balanced", ""), "strategy": "class_weight_balanced"})
    results.append(summary)

cv_results_df = pd.DataFrame(results).sort_values(by=["f1_mean", "roc_auc_mean"], ascending=False)
cv_results_df.reset_index(drop=True, inplace=True)
cv_results_df.head(12)

In [ ]:
plt.figure(figsize=(12, 5))
plot_df = cv_results_df.copy()
plot_df["label"] = plot_df["model"] + " | " + plot_df["strategy"]
sns.barplot(data=plot_df, x="f1_mean", y="label", orient="h")
plt.title("Cross-validated F1 by Model and Imbalance Strategy")
plt.xlabel("Mean F1")
plt.ylabel("Model | Strategy")
plt.tight_layout()
plt.show()

## Pick the best strategy

In [ ]:
best_row = cv_results_df.iloc[0]
best_model_name = best_row["model"]
best_strategy = best_row["strategy"]

print("Selected model:", best_model_name)
print("Selected strategy:", best_strategy)
print("CV F1 mean:", round(best_row["f1_mean"], 4))
print("CV ROC-AUC mean:", round(best_row["roc_auc_mean"], 4))

## Hyperparameter tuning

In [ ]:
all_models = get_model_configs()
selected_key = best_model_name if best_strategy != "class_weight_balanced" else f"{best_model_name}_balanced"
selected_model = clone(all_models[selected_key])

if best_strategy in {"smote", "smote_tomek"}:
    selected_model = clone_for_resampling(selected_model, best_strategy)
    sampler = get_resampling_strategies()[best_strategy]
    tune_estimator = Pipeline([("sampler", sampler), ("model", selected_model)])
    param_distributions = {f"model__{k}": v for k, v in get_hyperparam_grid(best_model_name).items()}
else:
    tune_estimator = selected_model
    param_distributions = get_hyperparam_grid(best_model_name)

random_search = RandomizedSearchCV(
    estimator=tune_estimator,
    param_distributions=param_distributions,
    n_iter=50,
    scoring="f1",
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=42),
    n_jobs=-1,
    random_state=42,
    verbose=0,
)
random_search.fit(x_train, y_train)

best_estimator = random_search.best_estimator_
print("Best CV F1 from random search:", round(random_search.best_score_, 4))
print("Best params:")
print(random_search.best_params_)

## Final test evaluation

In [ ]:
test_metrics = evaluate_on_test(best_estimator, x_test, y_test)

print("Test ROC-AUC:", round(test_metrics["roc_auc"], 4))
print("Test PR-AUC:", round(test_metrics["pr_auc"], 4))
print("Test F1:", round(test_metrics["f1"], 4))
print("Test Precision:", round(test_metrics["precision"], 4))
print("Test Recall:", round(test_metrics["recall"], 4))

print("\nClassification report (default 0.5 threshold):")
print(classification_report(y_test, test_metrics["y_pred"], digits=4))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
ConfusionMatrixDisplay(test_metrics["confusion_matrix"]).plot(ax=axes[0], colorbar=False)
axes[0].set_title("Confusion Matrix")
RocCurveDisplay.from_estimator(best_estimator, x_test, y_test, ax=axes[1])
axes[1].set_title("ROC Curve")
PrecisionRecallDisplay.from_estimator(best_estimator, x_test, y_test, ax=axes[2])
axes[2].set_title("Precision-Recall Curve")
plt.tight_layout()
plt.show()

## Threshold tuning

In [ ]:
# Tune threshold on cross-validated training predictions to avoid contaminating the held-out test set.
y_proba_cv = cross_val_predict(
    best_estimator,
    x_train,
    y_train,
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=42),
    method="predict_proba",
    n_jobs=-1,
)[:, 1]
optimal_threshold = find_optimal_threshold_from_proba(y_train, y_proba_cv)

# Apply the calibrated threshold to the test set for final evaluation.
y_proba = best_estimator.predict_proba(x_test)[:, 1]
y_pred_opt = (y_proba >= optimal_threshold).astype(int)

print("Optimal threshold (tuned on train CV):", round(optimal_threshold, 4))
print("Metrics at optimal threshold:")
print("F1:", round(f1_score(y_test, y_pred_opt), 4))
print("Precision:", round(precision_score(y_test, y_pred_opt, zero_division=0), 4))
print("Recall:", round(recall_score(y_test, y_pred_opt, zero_division=0), 4))
print("\nClassification report (optimal threshold):")
print(classification_report(y_test, y_pred_opt, digits=4))

## Save model + metadata

In [ ]:
models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(best_estimator, models_dir / "best_model.joblib")
joblib.dump(float(optimal_threshold), models_dir / "best_threshold.joblib")

summary = {
    "best_model": best_model_name,
    "best_strategy": best_strategy,
    "cv_f1_mean": float(best_row["f1_mean"]),
    "test_f1_default": float(test_metrics["f1"]),
    "test_roc_auc": float(test_metrics["roc_auc"]),
    "test_pr_auc": float(test_metrics["pr_auc"]),
    "optimal_threshold": float(optimal_threshold),
}
pd.Series(summary).to_json(models_dir / "model_summary.json", indent=2)

cv_results_df.to_csv(models_dir / "model_cv_results.csv", index=False)

print("Saved:")
print("-", models_dir / "best_model.joblib")
print("-", models_dir / "best_threshold.joblib")
print("-", models_dir / "model_summary.json")
print("-", models_dir / "model_cv_results.csv")